**This code is to use shape descriptors for classification using multi-layer perceptrons (MLPs):**

In [5]:
# Model training hyperparameters
MODEL = "pasc_ann"
DEVICE_ID = 1
SEED = 42
BATCH_SIZE = 512
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-5
DROPOUT = 0.3
OPTIMIZER = "AdamW"
LOG_DIR = f"./{MODEL}_logs/"
DATA_DIR = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v5/"

In [2]:
import os
import json
import numpy as np
import pandas as pd
import random
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
# --- Replace NaNs within each species group ---
def fill_by_class_mean(df, class_col="species"):
    df = df.replace(0, np.nan)
    df = df.dropna(axis=1, how='all')
    df_numeric = df.select_dtypes(include=[np.number])
    # Fill NaNs in numeric columns using the class-wise mean
    df[df_numeric.columns] = df.groupby(class_col)[df_numeric.columns].transform(
        lambda x: x.fillna(x.mean())
    )
    # Step 2: fill any remaining NaNs globally (for columns that were all NaN in a class)
    df[df_numeric.columns] = df[df_numeric.columns].fillna(df[df_numeric.columns].mean())
    return df

In [12]:
train_species_df = pd.read_csv(os.path.join(DATA_DIR, 'train_aug_labels.csv'))
val_species_df = pd.read_csv(os.path.join(DATA_DIR, 'val_labels.csv'))
test_species_df = pd.read_csv(os.path.join(DATA_DIR, 'test_labels.csv'))
train_species_df.shape, val_species_df.shape, test_species_df.shape

((34722, 2), (1158, 2), (771, 2))

In [13]:
train_feat_path = r"./bee_gt_shape_features_concise/bee_gt_features_train.csv"
val_feat_path   = r"./bee_gt_shape_features_concise/bee_gt_features_val.csv"
test_feat_path = r"./bee_gt_shape_features_concise/bee_gt_features_test.csv"
train_feat_df = pd.read_csv(train_feat_path) 
val_feat_df = pd.read_csv(val_feat_path) 
test_feat_df = pd.read_csv(test_feat_path)
train_feat_df.shape, val_feat_df.shape, test_feat_df.shape

((34722, 938), (1158, 938), (771, 938))

In [15]:
train_df = pd.merge(train_feat_df, train_species_df, on='image')
val_df = pd.merge(val_feat_df, val_species_df, on='image')
test_df = pd.merge(test_feat_df, test_species_df, on='image')
train_df.shape, val_df.shape, test_df.shape

((34722, 939), (1158, 939), (771, 939))

In [16]:
train_df.head(2)

,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area,species
0,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4330.0,608.6173,3.586139,0.248822,0.568018,0.960334,1.295621,0.146896,0.721149,...,6.401090,60.094750,27.64035,41.000250,0.143003,23.202300,0.120566,0.843097,0.036337,Bombus_impatiens
1,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4325.0,606.8244,3.585948,0.248535,0.568854,0.960330,1.296722,0.147594,0.721134,...,6.392323,60.289352,24.56276,43.832634,0.142942,14.504794,0.117950,0.825161,0.056889,Bombus_impatiens


In [17]:
val_df.head(2)

,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area,species
0,00903Q80CQ70TQX0AR403QU0Q050BRZQTQJKWR70JQN0YQ...,5410.0,434.57568,1.484547,0.497243,0.739172,0.739091,-1.247538,0.359978,0.326394,...,6.574201,69.938730,28.917168,43.730812,0.180586,16.606430,0.145540,0.805929,0.048531,Bombus_flavifrons
1,0090L09000N0Q0SQQ080FQFKCQHQ1R3K1RIQH0N0L0SQOR...,2058.0,449.34525,1.546949,0.172564,0.321864,0.762970,0.208325,0.128084,0.353566,...,4.861553,66.378494,25.980085,47.686066,0.129605,1.430412,0.070873,0.546835,0.382292,Bombus_terricola


In [18]:
test_df.head(2)

,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area,species
0,0090L0W0TQIQURLQ9RSQ3RKQTRQQOR7Q3R60CQG0OQ80FQ...,4279.0,506.32693,1.777138,0.443144,0.666926,0.826659,-1.310267,0.209744,0.437298,...,6.121056,63.953552,29.595713,47.932377,0.198838,1.826825,0.113867,0.572660,0.313473,Bombus_crotchii
1,0090Q0W0H0X0YQYKWR40S0SQURI0JQ70ARZQAR70CQHQUR...,2892.0,221.48022,1.697738,0.708824,0.953511,0.808119,-0.267619,0.740864,0.410981,...,7.187157,72.190110,29.266737,54.327305,0.088424,8.546120,0.073354,0.829575,0.097070,Bombus_vagans


In [19]:
train_df = fill_by_class_mean(train_df, class_col="species")
val_df   = fill_by_class_mean(val_df, class_col="species")
test_df = fill_by_class_mean(test_df, class_col="species")

X_train = train_df.drop(columns=["image", "species"])
X_val = val_df.drop(columns=["image", "species"])
X_test = test_df.drop(columns=["image", "species"])
y_train = train_df["species"]
y_val = val_df["species"]
y_test = test_df["species"]

# --- Sanity check ---
assert not X_train.isna().any().any(), "NaNs remain in X_train!"
assert not X_val.isna().any().any(), "NaNs remain in X_val!"
assert not X_test.isna().any().any(), "NaNs remain in X_val!"
print("✅ Class-wise NaN imputation complete — no missing values remain.")

✅ Class-wise NaN imputation complete — no missing values remain.


In [20]:
# ----------------- Standardize (important!) -----------------
scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train.values)
X_val_np   = scaler.transform(X_val.values)
X_test_np = scaler.transform(X_test.values)

# ----------------- Class mapping (use same mapping for train & val) -----------------
classes = sorted(y_train.unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
num_classes = len(classes)

y_train_idx = y_train.map(class_to_idx).values
y_val_idx   = y_val.map(class_to_idx).values
y_test_idx = y_test.map(class_to_idx).values

In [21]:
# ----------------- PyTorch Dataset -----------------
class ShapeFeatureDataset(Dataset):
    def __init__(self, X_np, y_np):
        self.X = torch.tensor(X_np, dtype=torch.float32)
        self.y = torch.tensor(y_np, dtype=torch.long)
    def __len__(self): 
        return len(self.X)
    def __getitem__(self, idx): 
        return self.X[idx], self.y[idx]

In [22]:
train_dataset = ShapeFeatureDataset(X_train_np, y_train_idx)
val_dataset   = ShapeFeatureDataset(X_val_np,   y_val_idx)
test_dataset  = ShapeFeatureDataset(X_test_np,   y_test_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

In [23]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
        # simple init
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

In [24]:
device = torch.device(f"cuda:{DEVICE_ID}")

input_dim = X_train.shape[1]
num_classes = len(classes)

model = MLPClassifier(input_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

In [25]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * X.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, log_dir):
    # Full training loop with logging and checkpointing 
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoints_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Initialize validation loss and start training
    best_val_loss = float("inf")
    print("Starting the first epoch...")

    for epoch in range(epochs):
        # ---- Training ----
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # ---- Validation ----
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # ---- Scheduler ----
        scheduler.step(val_loss)

        # ---- Logging ----
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- Checkpoint ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(checkpoints_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)

            checkpoint_path = os.path.join(checkpoints_dir, f"checkpoint_epoch{epoch+1}.pth")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            }, checkpoint_path)
        
        print(f"\r Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")    
    print("Training complete!")

    # ---- Save logs ----
    df = pd.DataFrame(history)
    csv_logfile = os.path.join(log_dir, "training_log.csv")
    json_logfile = os.path.join(log_dir, "training_log.json")

    df.to_csv(csv_logfile, index=False)
    with open(json_logfile, "w") as f:
        json.dump(history, f, indent=4)

    return history

In [26]:
%%time

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=EPOCHS,
    log_dir=LOG_DIR
)

Starting the first epoch...
 Epoch 1/100 | Train Loss: 3.9628 | Train Acc: 0.1978 | Val Loss: 3.7472 | Val Acc: 0.2228
 Epoch 2/100 | Train Loss: 2.8573 | Train Acc: 0.4253 | Val Loss: 3.6248 | Val Acc: 0.2435
 Epoch 3/100 | Train Loss: 2.2699 | Train Acc: 0.6065 | Val Loss: 3.6022 | Val Acc: 0.2660
 Epoch 4/100 | Train Loss: 1.8704 | Train Acc: 0.7422 | Val Loss: 3.6195 | Val Acc: 0.2530
 Epoch 5/100 | Train Loss: 1.6020 | Train Acc: 0.8383 | Val Loss: 3.6400 | Val Acc: 0.2634
 Epoch 6/100 | Train Loss: 1.4289 | Train Acc: 0.8970 | Val Loss: 3.6671 | Val Acc: 0.2608
 Epoch 7/100 | Train Loss: 1.3054 | Train Acc: 0.9386 | Val Loss: 3.6751 | Val Acc: 0.2737
 Epoch 8/100 | Train Loss: 1.2529 | Train Acc: 0.9544 | Val Loss: 3.6811 | Val Acc: 0.2686
 Epoch 9/100 | Train Loss: 1.2210 | Train Acc: 0.9634 | Val Loss: 3.6936 | Val Acc: 0.2599
 Epoch 10/100 | Train Loss: 1.1891 | Train Acc: 0.9726 | Val Loss: 3.7071 | Val Acc: 0.2625
 Epoch 11/100 | Train Loss: 1.1709 | Train Acc: 0.9768 | Val 

In [27]:
# ----------------- Load best model -----------------
# best_model_path = os.path.join(LOG_DIR, "checkpoints", "best_model.pth")
# model.load_state_dict(torch.load(best_model_path, map_location=device))
# model.to(device)
model.eval()

# ----------------- Evaluate on test set -----------------
def evaluate_test(model, loader, device):
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

test_acc = evaluate_test(model, test_loader, device)
print("Test accuracy:", test_acc)

Test accuracy: 0.24254215304798962
